In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in the current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Parse timestamp to timezone-aware datetime in UTC then convert to Asia/Shanghai (UTC+8)
# Keep original timestamp column as well
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Unique cmdb_id values from both files
unique_cmdb_metric = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()), name='metric_cmdb_ids')
unique_cmdb_trace = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()), name='trace_cmdb_ids')
unique_cmdb_both = pd.Series(sorted(pd.Index(unique_cmdb_metric).union(pd.Index(unique_cmdb_trace))), name='combined_cmdb_ids')

# Percentiles to compute
perc_list = [0.05, 0.25, 0.50, 0.75, 0.90, 0.95]
perc_names = ['P5','P25','P50','P75','P90','P95']

# 2) metric.csv summaries
# Group by kpi_name: count and percentiles over the entire series (across all cmdb_id)
metric_group = df_metric.groupby('kpi_name')['value']
metric_counts = metric_group.size().rename('count')
metric_quantiles = metric_group.quantile(perc_list).unstack(level=-1)
# quantile returned columns named by the quantile (e.g., 0.05). Rename to P5..P95
metric_quantiles.columns = perc_names
# Combine into one summary table
metric_summary = pd.concat([metric_counts, metric_quantiles], axis=1).reset_index()
# Sort by count desc, limit to top 20 for display
metric_summary_top20 = metric_summary.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# 3) trace.csv summaries
trace_group = df_trace.groupby('trace_name')['value']
trace_counts = trace_group.size().rename('count')
trace_quantiles = trace_group.quantile(perc_list).unstack(level=-1)
trace_quantiles.columns = perc_names
trace_summary = pd.concat([trace_counts, trace_quantiles], axis=1).reset_index()
trace_summary_top20 = trace_summary.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Keep outputs compact: show unique cmdb_id series and top-20 summary tables
unique_cmdb_metric, unique_cmdb_trace, unique_cmdb_both, metric_summary_top20, trace_summary_top20

```
Out[1]:
```
summary = (
    "Summary (plain English):\n\n"
    "1) Unique cmdb_id values\n"
    "- metric.csv: 56 unique cmdb_id values (examples: db_001, db_002, ..., redis_012)\n"
    "- trace.csv: 10 unique cmdb_id values (docker_001..docker_008, os_021, os_022)\n"
    "- combined (union): 56 unique cmdb_id values\n\n"
    "2) metric.csv (top KPI names by row count and global percentiles P5,P25,P50,P75,P90,P95)\n"
    "- CPU_iowait_time — count 1320 — P5 0.000000  P25 0.002779  P50 0.014623  P75 0.080656  P90 0.145331  P95 0.197465\n"
    "- CPU_system_time — count 1320 — P5 0.086152  P25 0.119527  P50 0.206310  P75 1.038530  P90 2.387515  P95 2.530666\n"
    "- Agent_ping — count 1320 — P5 1.000000  P25 1.000000  P50 1.000000  P75 1.000000  P90 1.000000  P95 1.000000\n"
    "- Outgoing_network_traffic — count 1320 — P5 0.000136  P25 0.001446  P50 0.010072  P75 0.059800  P90 0.095109  P95 1.836516\n"
    "- Page_pi — count 1320 — P5 0.000000  P25 0.000000  P50 0.000000  P75 0.000000  P90 0.000000  P95 0.675111\n"
    "- Page_po — count 1320 — P5 2.000000  P25 7.000000  P50 59.000000  P75 437.500000  P90 1556.300000  P95 1661.100000\n"
    "- CPU_idle_pct — count 1320 — P5 88.686720  P25 95.995579  P50 98.986156  P75 99.669272  P90 99.788792  P95 99.811038\n"
    "- Disk_await — count 1320 — P5 0.000000  P25 0.000000  P50 0.000000  P75 0.470000  P90 1.642000  P95 3.500000\n"
    "- Disk_avgqu_sz — count 1320 — P5 0.000000  P25 0.000832  P50 0.010966  P75 0.031628  P90 0.057090  P95 0.096399\n"
    "- CPU_user_time — count 1320 — P5 0.091707  P25 0.144528  P50 0.353050  P75 2.186969  P90 3.956440  P95 6.861810\n\n"
    "3) trace.csv (top trace_name by row count and global percentiles P5..P95)\n"
    "- trace.self.jdbc.row_count — count 240 — P5 404.8  P25 1432.0  P50 1560.0  P75 2320.5  P90 2497.1  P95 2610.05\n"
    "- trace.self.jdbc.error_rate — count 240 — P5 0.0  P25 0.0  P50 0.0  P75 0.0  P90 0.0  P95 0.0\n"
    "- trace.self.jdbc.duration_p95 — count 240 — P5 0.003  P25 0.003  P50 0.003  P75 0.003  P90 0.003  P95 0.003\n"
    "- trace.self.jdbc.duration_mean — count 240 — P5 0.001837  P25 0.002384  P50 0.002491  P75 0.002891  P90 0.003501  P95 0.003954\n"
    "- trace.self.local.row_count — count 240 — P5 243.8  P25 861.25  P50 939.0  P75 2030.0  P90 2184.0  P95 2282.0\n"
    "- trace.self.local.error_rate — count 240 — all percentiles 0.0\n"
    "- trace.self.local.duration_p95 — count 240 — P5 0.008  P25 0.009  P50 0.010425  P75 0.083  P90 0.092085  P95 0.094\n"
    "- trace.self.local.duration_mean — count 240 — P5 0.033045  P25 0.043906  P50 0.058653  P75 0.434204  P90 0.659691  P95 0.721124\n"
    "- trace.self.csf.row_count — count 180 — P5 130.6  P25 294.0  P50 532.0  P75 596.0  P90 632.0  P95 660.2\n"
    "- trace.self.csf.duration_p95 — count 180 — P5 0.091498  P25 0.099  P50 0.18715  P75 1.396125  P90 1.979165  P95 2.209337\n\n"
    "Notes: Counts are global row counts per KPI/trace_name (aggregated across all cmdb_id). "
    "Percentiles are computed across the full series for each KPI/trace_name (global thresholds) and shown above for the top items."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
51    redis_008
52    redis_009
53    redis_010
54    redis_011
55    redis_012
Name: metric_cmdb_ids, Length: 56, dtype: object, 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
Name: trace_cmdb_ids, dtype: object, 0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
51    redis_008
52    redis_009
53    redis_010
54    redis_011
55    redis_012
Name: combined_cmdb_ids, Length: 56, dtype: object,                     kpi_name  count         P5         P25         P50         P75          P90          P95
0            CPU_iowait_time   1320   0.000000    0.002779    0.014623    0.080656     0.145331     0.197465
1            CPU_system_time   1320   0.086152    0.119527    0.206310    1.038530     2.387515     2.530666
2                 Agent_ping   1320   1.000000    1.000000    1.000000    1.000000     1.000000     1.000000
3   Outgoing_network_traffic   1320   0.000136    0.001446    0.010072    0.059800     0.095109     1.836516
4                    Page_pi   1320   0.000000    0.000000    0.000000    0.000000     0.000000     0.675111
5                    Page_po   1320   2.000000    7.000000   59.000000  437.500000  1556.300000  1661.100000
6               CPU_idle_pct   1320  88.686720   95.995579   98.986156   99.669272    99.788792    99.811038
7                 Disk_await   1320   0.000000    0.000000    0.000000    0.470000     1.642000     3.500000
8              Disk_avgqu_sz   1320   0.000000    0.000832    0.010966    0.031628     0.057090     0.096399
9              CPU_user_time   1320   0.091707    0.144528    0.353050    2.186969     3.956440     6.861810
10              Disk_io_util   1320   0.000000    0.000000    0.000000    0.000000     0.011784     0.508376
11               Disk_wr_ios   1320   0.000000    1.000000    5.000000    7.000000    28.000000    38.000000
12                Disk_svctm   1320   0.000000    0.000000    0.000000    0.112500     1.264000     1.750000
13  Incoming_network_traffic   1320   0.000171    0.000934    0.008742    0.041280     1.006166     1.316430
14               Disk_wr_kbs   1320   0.002540    0.007404    0.057626    0.433688     1.528551     1.623237
15               Disk_rd_ios   1320   0.000000    0.000000    0.000000    0.000000     0.000000     0.000000
16           Memory_used_pct   1320   7.500000   16.870000   26.390000   44.620000    62.101000    76.300000
17               Memory_used   1320   3.504980    4.720623    6.693615    7.027652    30.361760    32.771192
18          Memory_available   1320   8.626283   11.552056   15.082065   30.151446    71.864875    72.486101
19                  ss_total   1320  70.950000  146.000000  368.500000  465.250000   654.000000   708.100000,                                            trace_name  count          P5          P25          P50          P75          P90          P95
0                           trace.self.jdbc.row_count    240  404.800000  1432.000000  1560.000000  2320.500000  2497.100000  2610.050000
1                          trace.self.jdbc.error_rate    240    0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
2                        trace.self.jdbc.duration_p95    240    0.003000     0.003000     0.003000     0.003000     0.003000     0.003000
3                       trace.self.jdbc.duration_mean    240    0.001837     0.002384     0.002491     0.002891     0.003501     0.003954
4                          trace.self.local.row_count    240  243.800000   861.250000   939.000000  2030.000000  2184.000000  2282.000000
5                         trace.self.local.error_rate    240    0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
6                       trace.self.local.duration_p95    240    0.008000     0.009000     0.010425     0.083000     0.092085     0.094000
7                      trace.self.local.duration_mean    240    0.005233     0.006010     0.006736     0.010606     0.014668     0.016107
8                            trace.self.csf.row_count    180  130.600000   294.000000   532.000000   596.000000   632.000000   660.200000
9                           trace.self.csf.error_rate    180    0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
10                        trace.self.csf.duration_p95    180    0.091498     0.099000     0.187150     1.396125     1.979165     2.209337
11                       trace.self.csf.duration_mean    180    0.033045     0.043906     0.058653     0.434204     0.659691     0.721124
12                 trace.self.flyremote.duration_mean    120    0.005424     0.007252     0.009272     0.010601     0.012751     0.014859
13                  trace.self.flyremote.duration_p95    120    0.006000     0.008000     0.011800     0.076650     0.089055     0.092620
14                    trace.self.flyremote.error_rate    120    0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
15                     trace.self.flyremote.row_count    120   37.000000   133.500000   145.000000   153.000000   163.000000   166.050000
16      trace.from_docker_003.remoteprocess.row_count     60   77.900000   259.000000   295.000000   319.250000   339.200000   349.150000
17     trace.from_docker_003.remoteprocess.error_rate     60    0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
18   trace.from_docker_003.remoteprocess.duration_p95     60    0.034142     0.036088     0.039000     0.040200     0.043100     0.046255
19  trace.from_docker_003.remoteprocess.duration_mean     60    0.024978     0.025881     0.026834     0.027627     0.028093     0.028514)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables if present; otherwise load
try:
    df_metric
    df_trace
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_trace = pd.read_csv('trace.csv')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts columns exist
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (UTC+8)
start_ts = pd.Timestamp('2020-05-27 05:30', tz=tz)
end_ts   = pd.Timestamp('2020-05-27 06:00', tz=tz)

# 1) Metrics: compute global P95 and P5 per (cmdb_id,kpi_name) using full series
metric_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile([0.95, 0.05])
    .unstack(level=-1)
    .rename(columns={0.95: 'global_P95', 0.05: 'global_P5'})
    .reset_index()
)

# Filter metric rows to the incident window
metric_window = df_metric[(df_metric['ts'] >= start_ts) & (df_metric['ts'] <= end_ts)].copy()

# Merge thresholds into window rows
metric_m = metric_window.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left')

# If there are rows in the window, compute breaches
if not metric_m.empty:
    metric_m['is_high'] = metric_m['value'] >= metric_m['global_P95']
    metric_m['is_low']  = metric_m['value'] <= metric_m['global_P5']

    def compute_metric_stats(group):
        gp95 = group['global_P95'].iloc[0]
        gp5  = group['global_P5'].iloc[0]
        bh = int(group['is_high'].sum())
        bl = int(group['is_low'].sum())
        total = bh + bl
        earliest = pd.NaT
        if total > 0:
            earliest = group.loc[group['is_high'] | group['is_low'], 'ts'].min()
        mx = group['value'].max()
        mn = group['value'].min()
        return pd.Series({
            'global_P95': gp95,
            'global_P5': gp5,
            'breach_count_high': bh,
            'breach_count_low': bl,
            'total_breach_count': total,
            'earliest_breach_timestamp': earliest,
            'max_value_in_window': mx,
            'min_value_in_window': mn
        })

    metric_stats = metric_m.groupby(['cmdb_id','kpi_name']).apply(compute_metric_stats).reset_index()

    # Keep only pairs with at least one breach
    metric_stats = metric_stats[metric_stats['total_breach_count'] > 0].copy()

    if not metric_stats.empty:
        # compute severities
        metric_stats['severity_high'] = np.where(
            metric_stats['breach_count_high']>0,
            np.where(metric_stats['global_P95']>0, metric_stats['max_value_in_window']/metric_stats['global_P95'], np.nan),
            np.nan
        )
        metric_stats['severity_low'] = np.where(
            metric_stats['breach_count_low']>0,
            np.where(metric_stats['min_value_in_window']>0, metric_stats['global_P5']/metric_stats['min_value_in_window'], np.nan),
            np.nan
        )
        # format earliest timestamp compactly
        metric_stats['earliest_breach_timestamp'] = metric_stats['earliest_breach_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        metric_breaches_top20 = metric_stats[[
            'cmdb_id','kpi_name','global_P95','global_P5',
            'breach_count_high','breach_count_low','total_breach_count',
            'earliest_breach_timestamp','max_value_in_window','min_value_in_window',
            'severity_high','severity_low'
        ]].sort_values(['total_breach_count','earliest_breach_timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)
    else:
        metric_breaches_top20 = pd.DataFrame([{
            'message': 'No P95/P5 breaches found in metric window (May 27 2020 05:30-06:00 UTC+8).'
        }])
else:
    metric_breaches_top20 = pd.DataFrame([{
        'message': 'No P95/P5 breaches found in metric window (May 27 2020 05:30-06:00 UTC+8).'
    }])

# 2) Traces: analogous procedure
trace_thresh = (
    df_trace
    .groupby(['cmdb_id','trace_name'])['value']
    .quantile([0.95, 0.05])
    .unstack(level=-1)
    .rename(columns={0.95: 'global_P95', 0.05: 'global_P5'})
    .reset_index()
)

trace_window = df_trace[(df_trace['ts'] >= start_ts) & (df_trace['ts'] <= end_ts)].copy()
trace_m = trace_window.merge(trace_thresh, on=['cmdb_id','trace_name'], how='left')

if not trace_m.empty:
    trace_m['is_high'] = trace_m['value'] >= trace_m['global_P95']
    trace_m['is_low']  = trace_m['value'] <= trace_m['global_P5']

    def compute_trace_stats(group):
        gp95 = group['global_P95'].iloc[0]
        gp5  = group['global_P5'].iloc[0]
        bh = int(group['is_high'].sum())
        bl = int(group['is_low'].sum())
        total = bh + bl
        earliest = pd.NaT
        if total > 0:
            earliest = group.loc[group['is_high'] | group['is_low'], 'ts'].min()
        mx = group['value'].max()
        mn = group['value'].min()
        return pd.Series({
            'global_P95': gp95,
            'global_P5': gp5,
            'breach_count_high': bh,
            'breach_count_low': bl,
            'total_breach_count': total,
            'earliest_breach_timestamp': earliest,
            'max_value_in_window': mx,
            'min_value_in_window': mn
        })

    trace_stats = trace_m.groupby(['cmdb_id','trace_name']).apply(compute_trace_stats).reset_index()
    trace_stats = trace_stats[trace_stats['total_breach_count'] > 0].copy()

    if not trace_stats.empty:
        trace_stats['severity_high'] = np.where(
            trace_stats['breach_count_high']>0,
            np.where(trace_stats['global_P95']>0, trace_stats['max_value_in_window']/trace_stats['global_P95'], np.nan),
            np.nan
        )
        trace_stats['severity_low'] = np.where(
            trace_stats['breach_count_low']>0,
            np.where(trace_stats['min_value_in_window']>0, trace_stats['global_P5']/trace_stats['min_value_in_window'], np.nan),
            np.nan
        )
        trace_stats['earliest_breach_timestamp'] = trace_stats['earliest_breach_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        trace_breaches_top20 = trace_stats[[
            'cmdb_id','trace_name','global_P95','global_P5',
            'breach_count_high','breach_count_low','total_breach_count',
            'earliest_breach_timestamp','max_value_in_window','min_value_in_window',
            'severity_high','severity_low'
        ]].sort_values(['total_breach_count','earliest_breach_timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)
    else:
        trace_breaches_top20 = pd.DataFrame([{
            'message': 'No P95/P5 breaches found in trace window (May 27 2020 05:30-06:00 UTC+8).'
        }])
else:
    trace_breaches_top20 = pd.DataFrame([{
        'message': 'No P95/P5 breaches found in trace window (May 27 2020 05:30-06:00 UTC+8).'
    }])

# Return compact tables (top-20) as variables
metric_breaches_top20, trace_breaches_top20

```
Out[2]:
```
summary = (
    "Summary (plain English):\n\n"
    "Metrics (metric.csv):\n"
    "- Multiple DB nodes (e.g., db_001..db_005 shown in the top results) had P95/P5 breaches during the incident window May 27 2020 05:30–06:00 (UTC+8).\n"
    "- Example KPIs with breaches: Hang, MEM_Total, On_Off_State, Row_Lock, SctRead_Per_Sec.\n"
    "- For those top entries each pair shows 60 total breaches in the 30-minute window (breach_count_high=30 and breach_count_low=30), with the earliest breach at 2020-05-27 05:30:00 CST.\n"
    "- For many KPIs max_value_in_window equals global_P95 (severity_high = 1.0) and min_value_in_window equals global_P5 (severity_low = 1.0). For KPIs whose global P95/P5 are zero (e.g., Row_Lock, SctRead_Per_Sec) severity is NaN because of zero thresholds.\n\n"
    "Traces (trace.csv):\n"
    "- Several docker containers (e.g., docker_001..docker_003 in the top results) show P95/P5 breaches in the same incident window.\n"
    "- Example trace_names with breaches: various error_rate metrics, and trace.self.jdbc.duration_p95 (and others).\n"
    "- Top trace pairs also show 60 total breaches (30 high, 30 low) with earliest breach at 2020-05-27 05:30:00 CST.\n"
    "- For numeric non-zero thresholds (e.g., trace.self.jdbc.duration_p95) severity_high = 1.0. For trace metrics with zero P95/P5 (e.g., many error_rate entries) severity is NaN.\n\n"
    "Conclusion:\n"
    "- P95/P5 breaches were found across many metric and trace (cmdb_id, KPI/trace_name) pairs during the incident window. Therefore we do not need to relax thresholds to P90/P10 for this window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id         kpi_name  global_P95   global_P5  breach_count_high  breach_count_low  total_breach_count earliest_breach_timestamp  max_value_in_window  min_value_in_window  severity_high  severity_low
0   db_001             Hang    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
1   db_001        MEM_Total  381.902264  381.902264                 30                30                  60   2020-05-27 05:30:00 CST           381.902264           381.902264            1.0           1.0
2   db_001     On_Off_State    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
3   db_001         Row_Lock    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
4   db_001  SctRead_Per_Sec    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
5   db_002             Hang    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
6   db_002        MEM_Total  381.902264  381.902264                 30                30                  60   2020-05-27 05:30:00 CST           381.902264           381.902264            1.0           1.0
7   db_002     On_Off_State    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
8   db_002         Row_Lock    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
9   db_002  SctRead_Per_Sec    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
10  db_003             Hang    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
11  db_003        MEM_Total  381.902264  381.902264                 30                30                  60   2020-05-27 05:30:00 CST           381.902264           381.902264            1.0           1.0
12  db_003     On_Off_State    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
13  db_003         Row_Lock    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
14  db_004             Hang    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
15  db_004        MEM_Total  381.902264  381.902264                 30                30                  60   2020-05-27 05:30:00 CST           381.902264           381.902264            1.0           1.0
16  db_004     On_Off_State    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0
17  db_004         Row_Lock    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
18  db_004  SctRead_Per_Sec    0.000000    0.000000                 30                30                  60   2020-05-27 05:30:00 CST             0.000000             0.000000            NaN           NaN
19  db_005             Hang    1.000000    1.000000                 30                30                  60   2020-05-27 05:30:00 CST             1.000000             1.000000            1.0           1.0,        cmdb_id                                    trace_name  global_P95  global_P5  breach_count_high  breach_count_low  total_breach_count earliest_breach_timestamp  max_value_in_window  min_value_in_window  severity_high  severity_low
0   docker_001    trace.from_os_022.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
1   docker_001                     trace.self.csf.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
2   docker_001               trace.self.flyremote.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
3   docker_001                  trace.self.jdbc.duration_p95       0.003      0.003                 30                30                  60   2020-05-27 05:30:00 CST                0.003                0.003            1.0           1.0
4   docker_001                    trace.self.jdbc.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
5   docker_001                   trace.self.local.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
6   docker_001  trace.to_docker_005.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
7   docker_001  trace.to_docker_006.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
8   docker_002    trace.from_os_022.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
9   docker_002                     trace.self.csf.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
10  docker_002               trace.self.flyremote.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
11  docker_002                  trace.self.jdbc.duration_p95       0.003      0.003                 30                30                  60   2020-05-27 05:30:00 CST                0.003                0.003            1.0           1.0
12  docker_002                    trace.self.jdbc.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
13  docker_002                   trace.self.local.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
14  docker_002  trace.to_docker_005.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
15  docker_002  trace.to_docker_006.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
16  docker_003    trace.from_os_021.remoteprocess.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
17  docker_003                     trace.self.csf.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
18  docker_003               trace.self.flyremote.error_rate       0.000      0.000                 30                30                  60   2020-05-27 05:30:00 CST                0.000                0.000            NaN           NaN
19  docker_003                  trace.self.jdbc.duration_p95       0.003      0.003                 30                30                  60   2020-05-27 05:30:00 CST                0.003                0.003            1.0           1.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables from kernel if present
try:
    df_metric
    metric_thresh
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # compute global thresholds if missing
    metric_thresh = (
        df_metric
        .groupby(['cmdb_id','kpi_name'])['value']
        .quantile([0.95, 0.05])
        .unstack(level=-1)
        .rename(columns={0.95: 'global_P95', 0.05: 'global_P5'})
        .reset_index()
    )

# Incident window (UTC+8)
start_ts = pd.Timestamp('2020-05-27 05:30', tz=tz)
end_ts   = pd.Timestamp('2020-05-27 06:00', tz=tz)

# Filter metric rows to the incident window
metric_window = df_metric[(df_metric['ts'] >= start_ts) & (df_metric['ts'] <= end_ts)].copy()

# Merge global thresholds (precomputed) into window rows
metric_w = metric_window.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left')

# Determine breaches (high and low). We'll create two sets (high and low) to detect runs separately.
metric_w['is_high'] = metric_w['value'] >= metric_w['global_P95']
metric_w['is_low']  = metric_w['value'] <= metric_w['global_P5']

# Function to detect consecutive-minute runs for a DataFrame filtered to one breach type
def detect_runs(df, breach_type_col_name):
    if df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','breach_type','fault_start_timestamp','fault_end_timestamp',
            'duration_minutes','data_point_count','max_value_in_fault','min_value_in_fault',
            'global_P95','global_P5','severity','significant'
        ])
    # df must be sorted by ts
    df = df.sort_values('ts').copy()
    # compute gap in seconds between consecutive rows
    df['gap_s'] = df['ts'].diff().dt.total_seconds().fillna(0)
    # start new segment when gap > 60 seconds (more than one minute gap)
    df['segment'] = (df['gap_s'] > 60).cumsum()
    # aggregate per segment
    agg = df.groupby(['cmdb_id','kpi_name','segment']).agg(
        fault_start_ts=('ts','min'),
        fault_end_ts=('ts','max'),
        data_point_count=('ts','size'),
        max_value_in_fault=('value','max'),
        min_value_in_fault=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first')
    ).reset_index(drop=False)
    # compute duration_minutes inclusive
    agg['duration_minutes'] = ((agg['fault_end_ts'] - agg['fault_start_ts']).dt.total_seconds() // 60).astype(int) + 1
    # breach_type assigned based on function arg
    agg['breach_type'] = breach_type_col_name
    # compute severity
    def calc_sev(row):
        if row['breach_type'] == 'high':
            gp95 = row['global_P95']
            if pd.isna(gp95) or gp95 == 0:
                return np.nan
            return row['max_value_in_fault'] / gp95
        else: # low
            gp5 = row['global_P5']
            if pd.isna(row['min_value_in_fault']) or row['min_value_in_fault'] == 0:
                return np.nan
            return gp5 / row['min_value_in_fault']
    agg['severity'] = agg.apply(calc_sev, axis=1)
    agg['significant'] = agg['severity'].fillna(0) >= 1.5
    # format timestamps
    agg['fault_start_timestamp'] = agg['fault_start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['fault_end_timestamp']   = agg['fault_end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    out = agg[[
        'cmdb_id','kpi_name','breach_type','fault_start_timestamp','fault_end_timestamp',
        'duration_minutes','data_point_count','max_value_in_fault','min_value_in_fault',
        'global_P95','global_P5','severity','significant'
    ]].copy()
    return out

# Build high breaches dataframe
high_df = metric_w[metric_w['is_high']].copy()
low_df  = metric_w[metric_w['is_low']].copy()

high_runs = detect_runs(high_df, 'high')
low_runs  = detect_runs(low_df, 'low')

# Combine runs (note: if a timestamp qualifies for both high and low it will appear in both lists)
faults = pd.concat([high_runs, low_runs], ignore_index=True)
# If no faults, return explicit indicator
if faults.empty:
    metric_faults_top20 = pd.DataFrame([{'message':'no faults found in metric window'}])
else:
    # sort by data_point_count desc then severity desc, return top 20
    metric_faults_top20 = faults.sort_values(['data_point_count','severity'], ascending=[False, False]).head(20).reset_index(drop=True)

# Display compact result variable
metric_faults_top20

```
Out[3]:
```
summary = (
    "Fault identification summary (metrics, incident window 2020-05-27 05:30–06:00 UTC+8):\n\n"
    "- Top 20 detected faults (one row per fault) all are high-side breaches (value >= global P95).\n"
    "- Each listed fault is a consecutive run lasting the full analyzed window: fault_start 2020-05-27 05:30, fault_end 2020-05-27 05:59, duration 30 minutes, data_point_count 30.\n"
    "- Two faults stand out as significant (severity >= 1.5):\n"
    "    * db_010 / ACS — severity 2.0 (max 0.02 vs P95 0.01) — significant: True\n"
    "    * db_010 / Sess_Active — severity 2.0 (max 0.20 vs P95 0.10) — significant: True\n"
    "- The remaining top faults (examples: db_001..db_006 Hang, MEM_Total, On_Off_State, Proc_User_Used_Pct, etc.) show severity = 1.0 (max equals global P95) and are not flagged as 'significant' by the 1.5 threshold.\n"
    "- No low-side faults appear in the returned top-20 results (i.e., no runs where value <= global P5 among top faults).\n\n"
    "Actionable recommendation:\n"
    "- Prioritize investigation of db_010 (ACS and Sess_Active) because they exceed the P95 by a large margin (severity 2.0).\n"
    "- Review the other DB KPIs (MEM_Total, Hang, On_Off_State) which were at or near their P95 for the entire window — they indicate sustained high values but not large exceedances.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id            kpi_name breach_type    fault_start_timestamp      fault_end_timestamp  duration_minutes  data_point_count  max_value_in_fault  min_value_in_fault  global_P95   global_P5  severity  significant
0   db_010                 ACS        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            0.020000            0.010000    0.010000    0.010000       2.0         True
1   db_010         Sess_Active        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            0.200000            0.100000    0.100000    0.100000       2.0         True
2   db_001                Hang        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
3   db_001           MEM_Total        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30          381.902264          381.902264  381.902264  381.902264       1.0        False
4   db_001        On_Off_State        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
5   db_002                Hang        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
6   db_002           MEM_Total        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30          381.902264          381.902264  381.902264  381.902264       1.0        False
7   db_002        On_Off_State        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
8   db_003                Hang        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
9   db_003           MEM_Total        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30          381.902264          381.902264  381.902264  381.902264       1.0        False
10  db_003        On_Off_State        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
11  db_004                Hang        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
12  db_004           MEM_Total        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30          381.902264          381.902264  381.902264  381.902264       1.0        False
13  db_004        On_Off_State        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
14  db_005                Hang        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
15  db_005           MEM_Total        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30          506.929929          506.929929  506.929929  506.929929       1.0        False
16  db_005        On_Off_State        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
17  db_005  Proc_User_Used_Pct        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            3.010000            3.010000    3.010000    3.010000       1.0        False
18  db_006                Hang        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30            1.000000            1.000000    1.000000    1.000000       1.0        False
19  db_006           MEM_Total        high  2020-05-27 05:30:00 CST  2020-05-27 05:59:00 CST                30                30          508.252410          508.252410  508.252410  508.252410       1.0        False```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables if available
try:
    df_metric
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts column exists and timezone is set
if 'ts' not in df_metric.columns:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

target = 'db_010'
start_ts = pd.Timestamp('2020-05-27 05:30', tz=tz)
end_ts   = pd.Timestamp('2020-05-27 06:00', tz=tz)

# Filter full series for db_010
df_db010_full = df_metric[df_metric['cmdb_id'] == target].copy()
if df_db010_full.empty:
    result_db010 = pd.DataFrame([{'message': 'no metrics for db_010'}])
else:
    # 1) global thresholds per kpi_name using full series
    thresh = (
        df_db010_full
        .groupby('kpi_name')['value']
        .quantile([0.95, 0.05])
        .unstack(level=-1)
        .rename(columns={0.95: 'global_P95', 0.05: 'global_P5'})
        .reset_index()
    )

    # 2) windowed rows and merge thresholds for per-row comparison
    window = df_db010_full[(df_db010_full['ts'] >= start_ts) & (df_db010_full['ts'] <= end_ts)].copy()
    if not window.empty:
        window = window.merge(thresh, on='kpi_name', how='left')
        # per-kpi aggregates in window
        agg_window = window.groupby('kpi_name').agg(
            earliest_timestamp_in_window=('ts','min'),
            data_point_count=('ts','size'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            breach_count_high=('value', lambda s: int((s >= window.loc[s.index, 'global_P95']).sum())),
            breach_count_low =('value', lambda s: int((s <= window.loc[s.index, 'global_P5']).sum()))
        ).reset_index()
    else:
        # empty window aggregates
        agg_window = pd.DataFrame(columns=['kpi_name','earliest_timestamp_in_window','data_point_count',
                                           'max_value_in_window','min_value_in_window','breach_count_high','breach_count_low'])

    # 3) merge thresholds with window aggregates to ensure one row per kpi_name
    merged = thresh.merge(agg_window, on='kpi_name', how='left')

    # fill missing numeric values
    merged['data_point_count'] = merged['data_point_count'].fillna(0).astype(int)
    merged['max_value_in_window'] = merged['max_value_in_window'].astype(float)
    merged['min_value_in_window'] = merged['min_value_in_window'].astype(float)
    merged['breach_count_high'] = merged['breach_count_high'].fillna(0).astype(int)
    merged['breach_count_low']  = merged['breach_count_low'].fillna(0).astype(int)

    # format earliest timestamp
    merged['earliest_timestamp_in_window'] = pd.to_datetime(merged['earliest_timestamp_in_window'])
    merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].fillna('')

    # compute severities with zero-division protection
    def sev_high(row):
        gp95 = row['global_P95']
        mv = row['max_value_in_window']
        if pd.isna(mv) or pd.isna(gp95) or gp95 == 0:
            return np.nan
        return mv / gp95

    def sev_low(row):
        gp5 = row['global_P5']
        mn = row['min_value_in_window']
        if pd.isna(mn) or pd.isna(gp5) or mn == 0:
            return np.nan
        return gp5 / mn

    merged['severity_high'] = merged.apply(sev_high, axis=1)
    merged['severity_low']  = merged.apply(sev_low, axis=1)

    # 4) candidate_reason_hint
    def hint(k):
        kk = k.lower()
        if any(x in kk for x in ['cpu','proc']):
            return 'CPU'
        if any(x in kk for x in ['network','incoming','outgoing']):
            return 'network'
        if any(x in kk for x in ['sess','conn','session']):
            return 'db connection'
        if any(x in kk for x in ['mem','memory']):
            return 'memory'
        return ''

    merged['candidate_reason_hint'] = merged['kpi_name'].apply(hint)

    # prepare final table with cmdb_id column and sort by severity desc (max of two severities)
    merged['severity_sort'] = merged[['severity_high','severity_low']].fillna(0).max(axis=1)
    merged['cmdb_id'] = target

    result_db010 = merged[[
        'cmdb_id','kpi_name','global_P95','global_P5','earliest_timestamp_in_window','data_point_count',
        'breach_count_high','breach_count_low','max_value_in_window','min_value_in_window',
        'severity_high','severity_low','candidate_reason_hint','severity_sort'
    ]].sort_values('severity_sort', ascending=False).head(20).reset_index(drop=True)

# display compact result
result_db010

```
Out[4]:
```
Summary for db_010 (incident window 2020-05-27 05:30–06:00 UTC+8)

Key findings (one row per KPI was computed — shown below are the most important highlights):

- Highest severity exceedances (strongest signals)
  - LFParaWrite_Per_Sec — global_P95=0.35, global_P5=0.05, window: 30 points, max=5.91, breach_count_high=2, breach_count_low=6, severity_high ≈ 16.89. (very large exceedance)
  - LFSync_Per_Sec — global_P95=1.0605, global_P5=0.36, window: 30 points, max=13.88, breach_count_high=1, breach_count_low=1, severity_high ≈ 13.09. (very large exceedance)

- Strong DB-connection signals
  - ACS — global_P95=0.01, window:30 pts, max=0.02, breach_count_high=30, severity_high=2.0
  - Sess_Active — global_P95=0.10, window:30 pts, max=0.20, breach_count_high=30, severity_high=2.0 (candidate_reason_hint = "db connection")
  - Login_Per_Sec — severity_high=2.0, breach_count_high=10

- Sustained high but near-threshold KPIs (severity ≈ 1.0)
  - MEM_Used, MEM_Used_Pct, MEM_real_util — memory-related metrics slightly above P95 (severity ≈ 1.00 — candidate_reason_hint = "memory")
  - CPU_Used_Pct, Proc_User_Used_Pct, Proc_Used_Pct, CPU_free_pct — CPU-related metrics near P95 but only marginally above (severity ≈ 1.00 — candidate_reason_hint = "CPU")
  - Redo_Per_Sec shows large absolute values but its max is slightly below global_P95 (severity_high ≈ 0.94)

Other compact facts (from the per-KPI rows)
- Earliest timestamp in window for all listed KPIs: 2020-05-27 05:30:00 CST.
- Most KPIs have full coverage in the window (data_point_count = 30).
- Several KPIs show many high-side breaches (e.g., ACS and Sess_Active: 30 high breaches each).
- A few KPIs show low-side breaches as well (counts included per KPI).

Heuristic candidate_reason_hint summary (based on KPI names)
- db connection related: Sess_Active, Session_pct, ACS, Login_Per_Sec — strong indicators of connection/session pressure.
- memory related: MEM_Used, MEM_Used_Pct, MEM_real_util — modestly above baseline.
- CPU related: CPU_Used_Pct, Proc_* — marginally above baseline.
- No explicit "network" hints in the top sorted results.

Interpretation / short recommendation
- The strongest and most abnormal signals are LFParaWrite_Per_Sec and LFSync_Per_Sec (very large severities > 10). These point to unusually heavy write/sync activity (storage / I/O subsystem).
- Concomitant/session signals (Sess_Active, ACS, Login_Per_Sec) are also highly elevated (severity 2.0), suggesting session/connection pressure on the DB during the same window.
- Memory and CPU metrics are near their P95 but not strongly above it (severity ≈ 1.0).
- Suggested immediate focus: investigate storage I/O (disk latency/throughput and sync activity) and DB session/connection handling on db_010; these two areas are the strongest candidate root causes given the metric patterns.

If you want, I can now:
- produce the full compact table again sorted or filtered differently, or
- check trace metrics for db_010 to corroborate I/O or connection issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id             kpi_name   global_P95    global_P5 earliest_timestamp_in_window  data_point_count  breach_count_high  breach_count_low  max_value_in_window  min_value_in_window  severity_high  severity_low candidate_reason_hint  severity_sort
0   db_010  LFParaWrite_Per_Sec     0.350000     0.050000      2020-05-27 05:30:00 CST                30                  2                 6             5.910000             0.050000      16.885714      1.000000                            16.885714
1   db_010       LFSync_Per_Sec     1.060500     0.360000      2020-05-27 05:30:00 CST                30                  1                 1            13.880000             0.360000      13.088166      1.000000                            13.088166
2   db_010                  ACS     0.010000     0.010000      2020-05-27 05:30:00 CST                30                 30                28             0.020000             0.010000       2.000000      1.000000                             2.000000
3   db_010          Sess_Active     0.100000     0.100000      2020-05-27 05:30:00 CST                30                 30                29             0.200000             0.100000       2.000000      1.000000         db connection       2.000000
4   db_010        Login_Per_Sec     0.200000     0.100000      2020-05-27 05:30:00 CST                30                 10                20             0.400000             0.100000       2.000000      1.000000                             2.000000
5   db_010         Redo_Per_Sec   410.520000   180.025000      2020-05-27 05:30:00 CST                30                  0                 1           387.500000           137.100000       0.943925      1.313093                             1.313093
6   db_010          Session_pct     0.051000     0.031000      2020-05-27 05:30:00 CST                30                  3                 9             0.062000             0.031000       1.215686      1.000000         db connection       1.215686
7   db_010               DbTime    13.015000     8.300000      2020-05-27 05:30:00 CST                30                  1                 3            15.700000             8.100000       1.206300      1.024691                             1.206300
8   db_010         Exec_Per_Sec    82.300000    72.695000      2020-05-27 05:30:00 CST                30                  1                 3            82.400000            70.200000       1.001215      1.035541                             1.035541
9   db_010   Logic_Read_Per_Sec  8455.490000  3014.815000      2020-05-27 05:30:00 CST                30                  1                 2          8468.600000          2927.100000       1.001550      1.029967                             1.029967
10  db_010          TPS_Per_Sec   161.040000   146.160000      2020-05-27 05:30:00 CST                30                  1                 1           162.900000           145.400000       1.011550      1.005227                             1.011550
11  db_010         CPU_Used_Pct     2.638301     2.279795      2020-05-27 05:30:00 CST                26                  2                 1             2.668459             2.263033       1.011431      1.007407                   CPU       1.011431
12  db_010         Call_Per_Sec   444.190000   405.580000      2020-05-27 05:30:00 CST                30                  1                 1           445.900000           401.400000       1.003850      1.010414                             1.010414
13  db_010   Proc_User_Used_Pct     4.980000     4.900000      2020-05-27 05:30:00 CST                30                  0                10             4.960000             4.880000       0.995984      1.004098                   CPU       1.004098
14  db_010        Proc_Used_Pct     4.980000     4.900000      2020-05-27 05:30:00 CST                30                  0                10             4.960000             4.880000       0.995984      1.004098                   CPU       1.004098
15  db_010             MEM_Used   337.853209   337.787160      2020-05-27 05:30:00 CST                30                  1                 0           338.373480           337.794234       1.001540      0.999979                memory       1.001540
16  db_010         MEM_Used_Pct    66.359368    66.318464      2020-05-27 05:30:00 CST                30                  2                 0            66.433807            66.320226       1.001122      0.999973                memory       1.001122
17  db_010         CPU_free_pct    97.737057    97.362684      2020-05-27 05:30:00 CST                30                  1                 2            97.738776            97.331541       1.000018      1.000320                   CPU       1.000320
18  db_010        MEM_real_util    57.400500    57.380000      2020-05-27 05:30:00 CST                30                  1                24            57.410000            57.380000       1.000166      1.000000                memory       1.000166
19  db_010                  AWS     0.010000     0.000000      2020-05-27 05:30:00 CST                30                  5                25             0.010000             0.000000       1.000000           NaN                             1.000000```
```